### Intelligent Road Safety Analytics: Accident Severity Prediction and Risk Zone Scoring using Big Data Pipeline

### Spark SQL Analysis

### Objective
The objective of this notebook is to analyze the cleaned road accident dataset using Spark SQL. The analysis aims to identify accident patterns, severity trends, weather impacts, road conditions, vehicle involvement, and accident-prone locations to support data-driven road safety decisions.

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = (
    SparkSession.builder
    .appName("Road Safety Spark SQL Analysis")
    .master("local[*]")
    .getOrCreate()
)

26/07/30 17:14:16 WARN Utils: Your hostname, LAPTOP-S89A4J3G resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/30 17:14:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/30 17:14:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/30 17:14:18 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
df = spark.read.csv(
    "../cleaned_data/road_accident_cleaned",
    header=True,
    inferSchema=True
)

In [5]:
print("Rows :", df.count())
print("Columns :", len(df.columns))

df.show(5)

Rows : 300491
Columns : 23
+--------------+-------------+-----+-----------+----+--------------------+--------------------+-----------------+---------+----------------+--------------------------+-------------------+---------+--------------------+------------------+-------------------+-----------------------+------------------+-----------+-------------------+-------------------+------------------+------------+
|Accident_Index|Accident Date|Month|Day_of_Week|Year|    Junction_Control|     Junction_Detail|Accident_Severity| Latitude|Light_Conditions|Local_Authority_(District)|Carriageway_Hazards|Longitude|Number_of_Casualties|Number_of_Vehicles|       Police_Force|Road_Surface_Conditions|         Road_Type|Speed_limit|               Time|Urban_or_Rural_Area|Weather_Conditions|Vehicle_Type|
+--------------+-------------+-----+-----------+----+--------------------+--------------------+-----------------+---------+----------------+--------------------------+-------------------+---------+------

### Create Spark SQL Temporary View

The cleaned dataset is registered as a temporary SQL view to enable SQL-based analysis using Spark SQL.

In [7]:
df.createOrReplaceTempView("road_accidents")

In [8]:
spark.sql("SHOW TABLES").show()

+---------+--------------+-----------+
|namespace|     tableName|isTemporary|
+---------+--------------+-----------+
|         |road_accidents|       true|
+---------+--------------+-----------+



In [9]:
spark.sql("""
SELECT COUNT(*) AS Total_Accidents
FROM road_accidents
""").show()

+---------------+
|Total_Accidents|
+---------------+
|         300491|
+---------------+



Accident Severity Distribution

This query calculates the number of accidents for each severity level.

In [11]:
spark.sql("""
SELECT
    Accident_Severity,
    COUNT(*) AS Total_Accidents
FROM road_accidents
GROUP BY Accident_Severity
ORDER BY Total_Accidents DESC
""").show()

+-----------------+---------------+
|Accident_Severity|Total_Accidents|
+-----------------+---------------+
|           Slight|         256516|
|          Serious|          40083|
|            Fatal|           3892|
+-----------------+---------------+



Monthly Accident Trend

This query analyzes the distribution of road accidents across different months to identify seasonal accident patterns.

In [14]:
monthly_accidents = spark.sql("""
SELECT
    Month,
    COUNT(*) AS Total_Accidents
FROM road_accidents
GROUP BY Month
ORDER BY
CASE Month
    WHEN 'Jan' THEN 1
    WHEN 'Feb' THEN 2
    WHEN 'Mar' THEN 3
    WHEN 'Apr' THEN 4
    WHEN 'May' THEN 5
    WHEN 'Jun' THEN 6
    WHEN 'Jul' THEN 7
    WHEN 'Aug' THEN 8
    WHEN 'Sep' THEN 9
    WHEN 'Oct' THEN 10
    WHEN 'Nov' THEN 11
    WHEN 'Dec' THEN 12
END
""")

monthly_accidents.show()

+-----+---------------+
|Month|Total_Accidents|
+-----+---------------+
|  Jan|          22699|
|  Feb|          21282|
|  Mar|          24915|
|  Apr|          23664|
|  May|          25553|
|  Jun|          26156|
|  Jul|          26358|
|  Aug|          24941|
|  Sep|          26159|
|  Oct|          27701|
|  Nov|          28341|
|  Dec|          22722|
+-----+---------------+



Day-wise Accident Distribution

In [13]:
spark.sql("""
SELECT
    Day_of_Week,
    COUNT(*) AS Total_Accidents
FROM road_accidents
GROUP BY Day_of_Week
ORDER BY Total_Accidents DESC
""").show()

+-----------+---------------+
|Day_of_Week|Total_Accidents|
+-----------+---------------+
|     Friday|          49286|
|  Wednesday|          45331|
|    Tuesday|          45220|
|   Thursday|          44495|
|     Monday|          42855|
|   Saturday|          40544|
|     Sunday|          32760|
+-----------+---------------+



Weather-wise Accident Analysis

In [15]:
weather_analysis = spark.sql("""
SELECT
    Weather_Conditions,
    COUNT(*) AS Total_Accidents
FROM road_accidents
GROUP BY Weather_Conditions
ORDER BY Total_Accidents DESC
""")

weather_analysis.show(truncate=False)

+---------------------+---------------+
|Weather_Conditions   |Total_Accidents|
+---------------------+---------------+
|Fine no high winds   |243330         |
|Raining no high winds|34732          |
|Other                |8752           |
|Snowing no high winds|4815           |
|Raining + high winds |3508           |
|Fine + high winds    |3136           |
|Fog or mist          |1681           |
|Snowing + high winds |537            |
+---------------------+---------------+



Road Surface Condition Analysis

In [16]:
road_surface_analysis = spark.sql("""
SELECT
    Road_Surface_Conditions,
    COUNT(*) AS Total_Accidents
FROM road_accidents
GROUP BY Road_Surface_Conditions
ORDER BY Total_Accidents DESC
""")

road_surface_analysis.show(truncate=False)

+-----------------------+---------------+
|Road_Surface_Conditions|Total_Accidents|
+-----------------------+---------------+
|Dry                    |203267         |
|Wet or damp            |80298          |
|Frost or ice           |11873          |
|Snow                   |4691           |
|Flood over 3cm. deep   |362            |
+-----------------------+---------------+



Vehicle Type Involvement


In [17]:
vehicle_analysis = spark.sql("""
SELECT
    Vehicle_Type,
    COUNT(*) AS Total_Accidents
FROM road_accidents
GROUP BY Vehicle_Type
ORDER BY Total_Accidents DESC
""")

vehicle_analysis.show(truncate=False)

+-------------------------------------+---------------+
|Vehicle_Type                         |Total_Accidents|
+-------------------------------------+---------------+
|Car                                  |233961         |
|Van / Goods 3.5 tonnes mgw or under  |15312          |
|Motorcycle over 500cc                |10959          |
|Bus or coach (17 or more pass seats) |8497           |
|Motorcycle 125cc and under           |6687           |
|Goods 7.5 tonnes mgw and over        |6354           |
|Taxi/Private hire car                |5403           |
|Motorcycle 50cc and under            |3618           |
|Motorcycle over 125cc and up to 500cc|3217           |
|Other vehicle                        |2448           |
|Goods over 3.5t. and under 7.5t      |2447           |
|Minibus (8 - 16 passenger seats)     |804            |
|Agricultural vehicle                 |716            |
|Pedal cycle                          |65             |
|Ridden horse                         |3        

Speed Limit vs Accident Severity

In [18]:
speed_severity = spark.sql("""
SELECT
    Speed_limit,
    Accident_Severity,
    COUNT(*) AS Total_Accidents
FROM road_accidents
GROUP BY Speed_limit, Accident_Severity
ORDER BY Speed_limit, Accident_Severity
""")

speed_severity.show(truncate=False)

+-----------+-----------------+---------------+
|Speed_limit|Accident_Severity|Total_Accidents|
+-----------+-----------------+---------------+
|10         |Slight           |2              |
|15         |Slight           |2              |
|20         |Fatal            |13             |
|20         |Serious          |368            |
|20         |Slight           |2407           |
|30         |Fatal            |1516           |
|30         |Serious          |23616          |
|30         |Slight           |169379         |
|40         |Fatal            |363            |
|40         |Serious          |3256           |
|40         |Slight           |21544          |
|50         |Fatal            |218            |
|50         |Serious          |1450           |
|50         |Slight           |8331           |
|60         |Fatal            |1290           |
|60         |Serious          |8720           |
|60         |Slight           |36046          |
|70         |Fatal            |492      

Urban vs Rural Accident Analysis

In [19]:
urban_rural = spark.sql("""
SELECT
    Urban_or_Rural_Area,
    COUNT(*) AS Total_Accidents
FROM road_accidents
GROUP BY Urban_or_Rural_Area
ORDER BY Total_Accidents DESC
""")

urban_rural.show()

+-------------------+---------------+
|Urban_or_Rural_Area|Total_Accidents|
+-------------------+---------------+
|              Urban|         193340|
|              Rural|         107151|
+-------------------+---------------+



Top 10 Accident-Prone Districts

In [21]:
risk_zones = spark.sql("""
SELECT
    `Local_Authority_(District)` AS District,
    COUNT(*) AS Total_Accidents
FROM road_accidents
GROUP BY `Local_Authority_(District)`
ORDER BY Total_Accidents DESC
LIMIT 10
""")

risk_zones.show(truncate=False)

+-----------+---------------+
|District   |Total_Accidents|
+-----------+---------------+
|Birmingham |5948           |
|Leeds      |4091           |
|Bradford   |2990           |
|Manchester |2869           |
|Westminster|2801           |
|Sheffield  |2684           |
|Cornwall   |2546           |
|Liverpool  |2536           |
|Barnet     |2286           |
|Lambeth    |2242           |
+-----------+---------------+

